# P4 — Foundation Models cho dự báo nhu cầu gián đoạn

Notebook triển khai ba time-series foundation models trên mẫu M5 gồm **Chronos-Bolt**, **TinyTimeMixer (TTM)** và **Moirai-2**.

Mỗi mô hình được đánh giá theo cùng ba cửa sổ rolling-origin. Với mỗi cửa sổ, mô hình chỉ nhận dữ liệu đến `train_end`, dự báo tiếp qua validation và test, sau đó chỉ giữ 28 ngày test. Đầu ra chuẩn gồm 9 mức quantile: `0.005, 0.025, 0.165, 0.25, 0.5, 0.75, 0.835, 0.975, 0.995`.

Phạm vi P4:
- zero-shot forecasting với Chronos-Bolt, TTM và Moirai-2;
- tạo probabilistic forecast phù hợp với từng mô hình;
- kiểm tra schema, quantile crossing và lưu raw forecast;
- benchmark runtime trong cùng môi trường chạy;
- thảo luận rủi ro rò rỉ dữ liệu pre-training.


## 01. Thiết lập môi trường


In [37]:
from pathlib import Path
import os, platform, random, subprocess, sys, time, warnings

CWD = Path.cwd()
ROOT = CWD if (CWD / "common/config.yaml").exists() else CWD.parent
assert (ROOT / "common/config.yaml").exists(), f"Không xác định được project root từ {CWD}"

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements.txt")], check=True)

import numpy as np
import pandas as pd
import torch
import yaml
from scipy import stats

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print("Python:", sys.version.split()[0], "| PyTorch:", torch.__version__, "| Device:", DEVICE)
print("System:", platform.platform())
print("Project root:", ROOT)


Python: 3.12.6 | PyTorch: 2.11.0 | Device: mps
System: macOS-15.6.1-arm64-arm-64bit
Project root: /Users/ntt2711/m5-intermittent-forecasting


## 02. Load cấu hình và dữ liệu


In [38]:
PATH_CONFIG = ROOT / "common/config.yaml"
PATH_WINDOWS = ROOT / "common/rolling_windows.csv"
PATH_SAMPLE = ROOT / "common/sample_ids.csv"
PATH_LONG = ROOT / "data/processed/data_long.parquet"
PATH_OUTPUT = ROOT / "data/outputs/forecasts"
PATH_OUTPUT.mkdir(parents=True, exist_ok=True)

for name, path in {"config": PATH_CONFIG, "rolling windows": PATH_WINDOWS, "sample ids": PATH_SAMPLE, "data_long": PATH_LONG}.items():
    assert path.exists(), f"Thiếu {name}: {path}"

with open(PATH_CONFIG, encoding="utf-8") as f: CFG = yaml.safe_load(f)

Q = [float(q) for q in CFG["quantiles"]]
H = int(CFG["horizon"])
FM = CFG["foundation"]
windows = pd.read_csv(PATH_WINDOWS)
sample_ids = pd.read_csv(PATH_SAMPLE)
long = pd.read_parquet(PATH_LONG)

print("data_long:", long.shape, "| sample:", sample_ids.shape, "| windows:", len(windows))
display(windows)


data_long: (6743034, 24) | sample: (3474, 1) | windows: 3


,window,train_end,val_end,test_start,test_end
0,W1,1885,1913,1914,1941
1,W2,1857,1885,1886,1913
2,W3,1829,1857,1858,1885


In [39]:
print(f"Series : {len(sample_ids):,}")
print(f"Days   : {long['d_int'].min()} → {long['d_int'].max()}")
print(f"Dates  : {long['date'].min()} → {long['date'].max()}")

if "segment" in long.columns:
    print("\nDemand segments:")
    display(long[["series_id", "segment"]].drop_duplicates()["segment"].value_counts())

Series : 3,474
Days   : 1 → 1941
Dates  : 2011-01-29 00:00:00 → 2016-05-22 00:00:00

Demand segments:


segment
lumpy           1000
intermittent    1000
smooth           980
erratic          494
Name: count, dtype: int64

## 03. Chuẩn bị dữ liệu và hàm kiểm tra chung


In [40]:
REQUIRED_COLS = {"series_id", "date", "d_int", "sales", "segment"}
missing = REQUIRED_COLS - set(long.columns)
assert not missing, f"data_long thiếu các cột: {sorted(missing)}"

p4_data = long[["series_id", "date", "d_int", "sales", "segment"]].copy()
p4_data["date"] = pd.to_datetime(p4_data["date"])
p4_data["d_int"] = pd.to_numeric(p4_data["d_int"]).astype(int)
p4_data["sales"] = pd.to_numeric(p4_data["sales"]).astype(np.float32)
p4_data = p4_data.sort_values(["series_id", "d_int"]).reset_index(drop=True)

expected_ids = set(sample_ids["series_id"].astype(str))
actual_ids = set(p4_data["series_id"].astype(str))
assert actual_ids == expected_ids, "series_id trong data_long không khớp sample_ids.csv"
assert p4_data.groupby("d_int")["date"].nunique().max() == 1

day_to_date = p4_data[["d_int", "date"]].drop_duplicates().set_index("d_int")["date"].to_dict()
series_ids = p4_data["series_id"].drop_duplicates().tolist()
series_groups = {sid: g for sid, g in p4_data.groupby("series_id", sort=False)}
N_SERIES = len(series_ids)
COLS = ["series_id", "date", "d_int", "quantile_level", "value", "model"]

print(f"Series: {N_SERIES:,} | d_int: {p4_data['d_int'].min()} → {p4_data['d_int'].max()}")
display(p4_data[["series_id", "segment"]].drop_duplicates()["segment"].value_counts())


Series: 3,474 | d_int: 1 → 1941


segment
lumpy           1000
intermittent    1000
smooth           980
erratic          494
Name: count, dtype: int64

In [41]:
def validate_forecast(df, model_name):
    expected_rows = len(windows) * N_SERIES * H * len(Q)
    assert list(df.columns) == COLS, f"{model_name}: sai schema"
    assert len(df) == expected_rows, f"{model_name}: {len(df):,} rows, expected {expected_rows:,}"
    assert df["series_id"].nunique() == N_SERIES
    assert df["quantile_level"].nunique() == len(Q)
    assert df["value"].notna().all() and np.isfinite(df["value"]).all() and (df["value"] >= 0).all()
    assert set(df["model"].unique()) == {model_name}
    assert df.duplicated(["series_id", "d_int", "quantile_level"]).sum() == 0
    print(f"{model_name}: {len(df):,} rows | {N_SERIES:,} series | {len(Q)} quantiles")

def check_quantile_crossing(df, model_name):
    wide = df.pivot(index=["series_id", "d_int"], columns="quantile_level", values="value")[sorted(Q)]
    n_crossing = int((np.diff(wide.to_numpy(), axis=1) < -1e-8).any(axis=1).sum())
    print(f"{model_name}: quantile crossing = {n_crossing:,}")
    assert n_crossing == 0


## 04. Chronos-Bolt — zero-shot

Chronos-Bolt được dùng trực tiếp từ checkpoint pretrained, không fine-tune trên M5. Context được cắt tại `train_end`; mô hình dự báo từ `train_end + 1` đến `test_end`, sau đó chỉ giữ 28 ngày test.

Chronos-Bolt hỗ trợ quantile native nhưng checkpoint chỉ được huấn luyện cho vùng quantile trung tâm. Khi yêu cầu các mức đuôi ngoài vùng hỗ trợ, thư viện có thể chặn chúng về biên được huấn luyện. Đây là hạn chế cần được giữ nguyên và ghi nhận ở raw forecast thay vì tự ngoại suy các đuôi cực trị.


In [42]:
from chronos import BaseChronosPipeline

CHRONOS_MODEL = FM["chronos_model"]
CHRONOS_CONTEXT = int(FM["chronos_context"])
CHRONOS_BATCH_SIZE = 64

chronos = BaseChronosPipeline.from_pretrained(CHRONOS_MODEL, device_map=DEVICE, torch_dtype=torch.float32)
print("Chronos-Bolt:", CHRONOS_MODEL, "| context:", CHRONOS_CONTEXT, "| device:", DEVICE)


INFO:p-4627:t-8310546624:_client.py:_send_single_request:HTTP Request: HEAD https://huggingface.co/amazon/chronos-bolt-base/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:p-4627:t-8310546624:_client.py:_send_single_request:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/amazon/chronos-bolt-base/5d9f166d69f47aef3401367a7b842e78fe97b121/config.json?%2Famazon%2Fchronos-bolt-base%2Fresolve%2Fmain%2Fconfig.json=&etag=%22a408d6913e164963361eeacbe10f2e62d894a1d7%22 "HTTP/1.1 200 OK"
INFO:p-4627:t-8310546624:_client.py:_send_single_request:HTTP Request: HEAD https://huggingface.co/amazon/chronos-bolt-base/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:p-4627:t-8310546624:_client.py:_send_single_request:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/amazon/chronos-bolt-base/5d9f166d69f47aef3401367a7b842e78fe97b121/config.json?%2Famazon%2Fchronos-bolt-base%2Fresolve%2Fmain%2Fconfig.json=&etag=%22a408d6913e164963361eeacbe10f

Chronos-Bolt: amazon/chronos-bolt-base | context: 512 | device: mps


In [43]:
chronos_outputs, chronos_runtime = [], []

for w_idx, w in windows.iterrows():
    window_name = str(w["window"]) if "window" in windows.columns else f"W{w_idx + 1}"
    train_end, test_start, test_end = int(w["train_end"]), int(w["test_start"]), int(w["test_end"])
    prediction_length = test_end - train_end
    test_offset = test_start - (train_end + 1)
    test_days = np.arange(test_start, test_end + 1)
    t0, window_parts = time.perf_counter(), []

    for batch_start in range(0, N_SERIES, CHRONOS_BATCH_SIZE):
        batch_ids = series_ids[batch_start:batch_start + CHRONOS_BATCH_SIZE]
        contexts = [torch.tensor(series_groups[sid].loc[series_groups[sid]["d_int"] <= train_end, "sales"].to_numpy(dtype=np.float32)[-CHRONOS_CONTEXT:], dtype=torch.float32) for sid in batch_ids]
        context_batch = torch.stack(contexts)

        with torch.inference_mode():
            q_pred, _ = chronos.predict_quantiles(inputs=context_batch, prediction_length=prediction_length, quantile_levels=Q)

        q_test = np.clip(q_pred.detach().cpu().numpy()[:, test_offset:test_offset + H, :], 0, None)
        assert q_test.shape == (len(batch_ids), H, len(Q))

        for i, sid in enumerate(batch_ids):
            block = pd.DataFrame({"series_id": np.repeat(sid, H * len(Q)), "d_int": np.repeat(test_days, len(Q)), "quantile_level": np.tile(Q, H), "value": q_test[i].reshape(-1)})
            block["date"] = block["d_int"].map(day_to_date); block["model"] = "chronos_bolt"
            window_parts.append(block[COLS])

    window_df = pd.concat(window_parts, ignore_index=True)
    elapsed = time.perf_counter() - t0
    chronos_outputs.append(window_df)
    chronos_runtime.append({"model": "chronos_bolt", "window": window_name, "seconds": elapsed})
    print(f"{window_name}: {len(window_df):,} rows | {elapsed:.1f} s")

chronos_all = pd.concat(chronos_outputs, ignore_index=True)
chronos_all["quantile_level"] = chronos_all["quantile_level"].astype(float)
chronos_all["value"] = chronos_all["value"].astype(np.float32)
validate_forecast(chronos_all, "chronos_bolt")
check_quantile_crossing(chronos_all, "chronos_bolt")

chronos_all.to_parquet(PATH_OUTPUT / "chronos_bolt.parquet", index=False)
pd.DataFrame(chronos_runtime).to_csv(PATH_OUTPUT / "chronos_bolt_runtime.csv", index=False)


W1: 875,448 rows | 33.4 s


W2: 875,448 rows | 29.8 s


W3: 875,448 rows | 30.3 s
chronos_bolt: 2,626,344 rows | 3,474 series | 9 quantiles
chronos_bolt: quantile crossing = 0


In [44]:
chronos_q_summary = chronos_all.groupby("quantile_level", as_index=False)["value"].mean()
display(chronos_q_summary)

chronos_wide = chronos_all.pivot(
    index=["series_id", "d_int"],
    columns="quantile_level",
    values="value"
)

print("q0.005 = q0.025:", f"{np.isclose(chronos_wide[0.005], chronos_wide[0.025]).mean():.2%}")
print("q0.975 = q0.995:", f"{np.isclose(chronos_wide[0.975], chronos_wide[0.995]).mean():.2%}")

,quantile_level,value
0,0.005,0.870251
1,0.025,0.870307
2,0.165,1.186231
3,0.250,1.584504
4,0.500,2.775701
5,0.750,4.687646
6,0.835,5.835121
7,0.975,6.987983
8,0.995,6.987983


q0.005 = q0.025: 97.59%
q0.975 = q0.995: 100.00%


### Nhận xét kết quả Chronos-Bolt

Chronos-Bolt tạo đầy đủ dự báo cho 3.474 chuỗi trong cả ba cửa sổ rolling-origin với 9 mức phân vị yêu cầu. Đầu ra sau xử lý không chứa giá trị âm và không xuất hiện hiện tượng quantile crossing.

Các phân vị được tạo trực tiếp từ cơ chế dự báo xác suất của Chronos-Bolt. Tuy nhiên, checkpoint chỉ hỗ trợ các mức phân vị trong khoảng được huấn luyện. Vì vậy, các phân vị cực biên nằm ngoài phạm vi này bị chặn về biên hỗ trợ của mô hình, thể hiện qua sự trùng nhau của một số phân vị ở hai đuôi.

Do đó, đầu ra đáp ứng cấu trúc probabilistic forecast yêu cầu nhưng khả năng biểu diễn phần đuôi của phân phối bị giới hạn bởi phạm vi phân vị native của checkpoint. Hạn chế này cần được xem xét khi đánh giá các chỉ tiêu phụ thuộc mạnh vào tail quantiles.

## 05. TinyTimeMixer (TTM) — zero-shot

TTM được dùng ở chế độ zero-shot và trả về point forecast. Mỗi context được chuẩn hóa theo mean/std tính **chỉ từ lịch sử đến `train_end`**, sau đó inverse-transform dự báo.

Để tạo 9 quantile theo schema chung, point forecast được bọc bằng phân phối Negative Binomial khi phương sai lịch sử lớn hơn trung bình; các trường hợp còn lại dùng Poisson. Vì vậy probabilistic forecast của TTM trong notebook này là **hậu xử lý**, không phải quantile native của checkpoint.


In [45]:
from tsfm_public.models.tinytimemixer import TinyTimeMixerForPrediction

TTM_MODEL = FM["ttm_model"]
TTM_CONTEXT = int(FM["ttm_context"])
TTM_BATCH_SIZE = 64

ttm = TinyTimeMixerForPrediction.from_pretrained(TTM_MODEL, num_input_channels=1).to(DEVICE)
ttm.eval()
print("TTM:", TTM_MODEL, "| context:", TTM_CONTEXT, "| device:", DEVICE)


INFO:p-4627:t-8310546624:_client.py:_send_single_request:HTTP Request: HEAD https://huggingface.co/ibm-granite/granite-timeseries-ttm-r2/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:p-4627:t-8310546624:_client.py:_send_single_request:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/ibm-granite/granite-timeseries-ttm-r2/d6a79570cac0f33d526601cd3a0fc7c80a8f9a2f/config.json?%2Fibm-granite%2Fgranite-timeseries-ttm-r2%2Fresolve%2Fmain%2Fconfig.json=&etag=%2209e937ef5082573c0785a180f23753110b78696c%22 "HTTP/1.1 200 OK"
INFO:p-4627:t-8310546624:_client.py:_send_single_request:HTTP Request: HEAD https://huggingface.co/ibm-granite/granite-timeseries-ttm-r2/resolve/main/model.safetensors "HTTP/1.1 302 Found"
Loading weights: 100%|██████████| 134/134 [00:00<00:00, 7390.26it/s]

TTM: ibm-granite/granite-timeseries-ttm-r2 | context: 512 | device: mps


In [46]:
def get_dispersion(data, train_end):
    mv = data.loc[data["d_int"] <= train_end].groupby("series_id")["sales"].agg(["mean", "var"])
    mean, var = mv["mean"].to_numpy(), mv["var"].to_numpy()
    r = np.full(len(mv), np.inf, dtype=np.float64)
    use_nb = (mean > 0) & (var > mean)
    r[use_nb] = mean[use_nb] ** 2 / (var[use_nb] - mean[use_nb])
    return dict(zip(mv.index, r))

def point_to_quantiles(point_df, model_name, disp_r):
    pf = point_df.copy()
    pf["value"] = pf["value"].clip(lower=0).astype(float)
    pf["r"] = pf["series_id"].map(disp_r).fillna(np.inf)
    mu, r = pf["value"].to_numpy(), pf["r"].to_numpy()
    finite = np.isfinite(r) & (r > 0)
    outputs = []

    for q in Q:
        qv = np.empty_like(mu, dtype=float)
        if finite.any():
            rr, mm = r[finite], mu[finite]
            p = rr / (rr + np.where(mm > 0, mm, 1e-9))
            qv[finite] = stats.nbinom.ppf(q, rr, p)
        if (~finite).any(): qv[~finite] = stats.poisson.ppf(q, mu[~finite])

        block = pf[["series_id", "date", "d_int"]].copy()
        block["quantile_level"] = float(q)
        block["value"] = np.nan_to_num(qv, nan=0.0, posinf=0.0, neginf=0.0)
        block["model"] = model_name
        outputs.append(block)

    return pd.concat(outputs, ignore_index=True)[COLS]


In [47]:
ttm_outputs, ttm_runtime = [], []

for w_idx, w in windows.iterrows():
    window_name = str(w["window"]) if "window" in windows.columns else f"W{w_idx + 1}"
    train_end, test_start, test_end = int(w["train_end"]), int(w["test_start"]), int(w["test_end"])
    prediction_length = test_end - train_end
    test_offset = test_start - (train_end + 1)
    test_days = np.arange(test_start, test_end + 1)
    disp_r = get_dispersion(p4_data, train_end)
    t0, window_points = time.perf_counter(), []

    for batch_start in range(0, N_SERIES, TTM_BATCH_SIZE):
        batch_ids = series_ids[batch_start:batch_start + TTM_BATCH_SIZE]
        contexts, means, stds = [], [], []

        for sid in batch_ids:
            y = series_groups[sid].loc[series_groups[sid]["d_int"] <= train_end, "sales"].to_numpy(dtype=np.float32)[-TTM_CONTEXT:]
            mean, std = float(y.mean()), float(y.std())
            if std < 1e-6: std = 1.0
            contexts.append((y - mean) / std); means.append(mean); stds.append(std)

        past_values = torch.tensor(np.stack(contexts), dtype=torch.float32).unsqueeze(-1).to(DEVICE)
        with torch.inference_mode(): outputs = ttm(past_values=past_values)

        point_scaled = outputs.prediction_outputs.detach().cpu().numpy()[..., 0]
        point_pred = point_scaled * np.asarray(stds, dtype=np.float32)[:, None] + np.asarray(means, dtype=np.float32)[:, None]
        point_test = np.clip(point_pred[:, :prediction_length], 0, None)[:, test_offset:test_offset + H]
        assert point_test.shape == (len(batch_ids), H)

        batch_df = pd.DataFrame({"series_id": np.repeat(batch_ids, H), "d_int": np.tile(test_days, len(batch_ids)), "value": point_test.reshape(-1)})
        batch_df["date"] = batch_df["d_int"].map(day_to_date)
        window_points.append(batch_df)

    point_window = pd.concat(window_points, ignore_index=True)
    window_df = point_to_quantiles(point_window, "ttm", disp_r)
    elapsed = time.perf_counter() - t0
    ttm_outputs.append(window_df)
    ttm_runtime.append({"model": "ttm", "window": window_name, "seconds": elapsed})
    print(f"{window_name}: {len(window_df):,} rows | {elapsed:.1f} s")

ttm_all = pd.concat(ttm_outputs, ignore_index=True)
ttm_all["quantile_level"] = ttm_all["quantile_level"].astype(float)
ttm_all["value"] = ttm_all["value"].astype(np.float32)
validate_forecast(ttm_all, "ttm")
check_quantile_crossing(ttm_all, "ttm")

ttm_all.to_parquet(PATH_OUTPUT / "ttm.parquet", index=False)
pd.DataFrame(ttm_runtime).to_csv(PATH_OUTPUT / "ttm_runtime.csv", index=False)


W1: 875,448 rows | 3.3 s
W2: 875,448 rows | 2.4 s
W3: 875,448 rows | 2.3 s
ttm: 2,626,344 rows | 3,474 series | 9 quantiles
ttm: quantile crossing = 0


In [48]:
ttm_wrapper_stats = []

for _, w in windows.iterrows():
    window_name = w["window"]
    train_end = int(w["train_end"])

    mv = long.loc[
        long["d_int"] <= train_end
    ].groupby("series_id")["sales"].agg(["mean", "var"])

    use_nb = (mv["mean"] > 0) & (mv["var"] > mv["mean"])

    ttm_wrapper_stats.append({
        "window": window_name,
        "negative_binomial": int(use_nb.sum()),
        "poisson": int((~use_nb).sum()),
        "nb_pct": use_nb.mean() * 100
    })

ttm_wrapper_stats = pd.DataFrame(ttm_wrapper_stats)
display(ttm_wrapper_stats)

,window,negative_binomial,poisson,nb_pct
0,W1,3472,2,99.942429
1,W2,3472,2,99.942429
2,W3,3470,4,99.884859


### Nhận xét kết quả TTM

TTM tạo dự báo đầy đủ cho 3.474 chuỗi trong cả ba cửa sổ rolling-origin với 9 mức phân vị và không xuất hiện quantile crossing trong đầu ra cuối cùng.

Khác với Chronos-Bolt, TTM trong thiết lập này cung cấp dự báo điểm. Để chuẩn hóa đầu ra về cùng định dạng dự báo xác suất, Negative Binomial được sử dụng khi trung bình lịch sử lớn hơn 0 và phương sai lớn hơn trung bình; các trường hợp còn lại sử dụng Poisson.

Thống kê theo từng cửa sổ cho thấy tiêu chí mean–variance của lớp bọc xác suất phân loại trên 99,8% chuỗi vào trường hợp overdispersion và sử dụng Negative Binomial, trong khi Poisson chỉ được áp dụng cho một số rất ít chuỗi.

Vì vậy, các phân vị cuối cùng không phải là đầu ra xác suất trực tiếp của TTM mà phụ thuộc vào giả định phân phối của lớp bọc. Khi đánh giá kết quả cần phân biệt giữa năng lực dự báo điểm của TTM và chất lượng của phương pháp chuyển đổi từ point forecast sang probabilistic forecast.

## 06. Moirai-2 — zero-shot

**LƯU Ý:** Moirai-2 được chạy trong môi trường Python riêng biệt vì phiên bản Uni2TS yêu cầu dependency khác với môi trường chính của Chronos/TTM.

Checkpoint `Salesforce/moirai-2.0-R-small` cung cấp 9 quantile native từ 0.1 đến 0.9. Các mức yêu cầu nằm trong khoảng này được nội suy tuyến tính; các mức ngoài khoảng được chặn tại biên 0.1/0.9. Sau mapping, **quantile rearrangement** sắp xếp các giá trị tại từng `(series_id, d_int)` để loại quantile crossing. Đây là bước hậu xử lý, không phải output native của Moirai-2.


In [49]:
import shutil

MOIRAI_ENV = ROOT / ".venv-moirai"
MOIRAI_PYTHON = MOIRAI_ENV / ("Scripts/python.exe" if os.name == "nt" else "bin/python")

def find_compatible_python():
    if (3, 10) <= sys.version_info[:2] <= (3, 12): return sys.executable
    for cmd in ("python3.12", "python3.11", "python3.10"):
        path = shutil.which(cmd)
        if path: return path
    if IN_COLAB:
        print("Colab đang dùng Python >= 3.13; cài Python 3.12 cho Moirai...")
        subprocess.run(["apt-get", "update", "-qq"], check=True)
        subprocess.run(["apt-get", "install", "-y", "python3.12", "python3.12-venv"], check=True)
        path = shutil.which("python3.12")
        if path: return path
    raise RuntimeError("Moirai/Uni2TS 2.0.0 cần Python 3.10–3.12. Hãy cài Python 3.12 rồi chạy lại cell này.")

def moirai_env_ok():
    if not MOIRAI_PYTHON.exists(): return False
    check = subprocess.run([
        str(MOIRAI_PYTHON), "-c",
        "import pyarrow, torch, uni2ts; from uni2ts.model.moirai2 import Moirai2Forecast, Moirai2Module"
    ], capture_output=True, text=True)
    return check.returncode == 0

if not moirai_env_ok():
    if MOIRAI_ENV.exists(): shutil.rmtree(MOIRAI_ENV)
    base_python = find_compatible_python()
    print("Đang tạo môi trường Moirai bằng:", base_python)
    subprocess.run([base_python, "-m", "venv", str(MOIRAI_ENV)], check=True)
    subprocess.run([str(MOIRAI_PYTHON), "-m", "pip", "install", "-q", "--upgrade", "pip"], check=True)
    subprocess.run([str(MOIRAI_PYTHON), "-m", "pip", "install", "-q", "uni2ts==2.0.0", "pyarrow"], check=True)

version = subprocess.check_output([str(MOIRAI_PYTHON), "--version"], text=True).strip()
device_check = subprocess.check_output([
    str(MOIRAI_PYTHON), "-c",
    "import torch; print(torch.__version__, '| CUDA:', torch.cuda.is_available(), '| MPS:', hasattr(torch.backends, 'mps') and torch.backends.mps.is_available())"
], text=True).strip()
print(f"Moirai environment ready | {version}")
print("Moirai Python:", MOIRAI_PYTHON)
print("Moirai PyTorch:", device_check)


Moirai environment ready | Python 3.12.6
Moirai Python: /Users/ntt2711/m5-intermittent-forecasting/.venv-moirai/bin/python
Moirai PyTorch: 2.4.1 | CUDA: False | MPS: True


In [50]:
moirai_code = r'''
from pathlib import Path
import time

import numpy as np
import pandas as pd
from gluonts.dataset.common import ListDataset
from uni2ts.model.moirai2 import Moirai2Forecast, Moirai2Module

ROOT = Path.cwd()
MODEL = "Salesforce/moirai-2.0-R-small"

CONTEXT = 512
BATCH_SIZE = 64
H = 28

Q = np.array([
    0.005, 0.025, 0.165, 0.25, 0.5,
    0.75, 0.835, 0.975, 0.995
])

NATIVE_Q = np.arange(0.1, 1.0, 0.1)

COLS = [
    "series_id",
    "date",
    "d_int",
    "quantile_level",
    "value",
    "model"
]

data = pd.read_parquet(
    ROOT / "data/processed/data_long.parquet"
)[["series_id", "date", "d_int", "sales"]].copy()

data["date"] = pd.to_datetime(data["date"])
data["d_int"] = data["d_int"].astype(int)
data["sales"] = data["sales"].astype("float32")
data["series_id"] = data["series_id"].astype(str)

sample_ids = pd.read_csv(ROOT / "common/sample_ids.csv")
ids = sample_ids["series_id"].astype(str).tolist()

data = (
    data[data["series_id"].isin(ids)]
    .sort_values(["series_id", "d_int"])
    .reset_index(drop=True)
)

groups = {
    sid: g
    for sid, g in data.groupby("series_id", sort=False)
}

assert len(groups) == len(ids), (
    f"Thiếu series trong data_long: expected {len(ids)}, found {len(groups)}"
)

dates = data[["d_int", "date"]].drop_duplicates()
windows = pd.read_csv(ROOT / "common/rolling_windows.csv")


# Load model
module = Moirai2Module.from_pretrained(MODEL)

all_outputs = []
runtime_rows = []
crossing_rows = []


# Rolling-origin forecasting
for _, w in windows.iterrows():
    window = str(w["window"])

    train_end = int(w["train_end"])
    test_start = int(w["test_start"])
    test_end = int(w["test_end"])

    pred_len = test_end - train_end
    t0 = time.perf_counter()

    entries = []

    for sid in ids:
        x = groups[sid].loc[
            groups[sid]["d_int"] <= train_end
        ].tail(CONTEXT)

        assert len(x) == CONTEXT

        entries.append({
            "start": pd.Period(x["date"].iloc[0], freq="D"),
            "target": x["sales"].to_numpy(dtype=np.float32),
            "item_id": sid
        })

    dataset = ListDataset(entries, freq="D")

    model = Moirai2Forecast(
        prediction_length=pred_len,
        target_dim=1,
        feat_dynamic_real_dim=0,
        past_feat_dynamic_real_dim=0,
        context_length=CONTEXT,
        module=module
    )

    forecasts = list(
        model.create_predictor(
            batch_size=BATCH_SIZE,
            device="auto"
        ).predict(dataset)
    )

    keep_from = test_start - (train_end + 1)
    test_days = np.arange(test_start, test_end + 1)

    rows = []
    crossing_before = 0
    crossing_after = 0
    n_points = 0

    # Quantile mapping + rearrangement
    for sid, fcst in zip(ids, forecasts):
        native = np.maximum(
            fcst.forecast_array[:, keep_from:keep_from + H],
            0
        )

        assert native.shape == (len(NATIVE_Q), H)

        for j, d_int in enumerate(test_days):
            # Nội suy/chặn các quantile yêu cầu từ native quantiles
            mapped = np.interp(
                Q,
                NATIVE_Q,
                native[:, j]
            )

            # Diagnostic trước rearrangement
            crossing_before += int(
                np.any(np.diff(mapped) < -1e-8)
            )

            # Quantile rearrangement
            values = np.sort(mapped)

            # Diagnostic sau rearrangement
            crossing_after += int(
                np.any(np.diff(values) < -1e-8)
            )

            n_points += 1

            rows.extend(
                (sid, d_int, float(q), float(value))
                for q, value in zip(Q, values)
            )

    out = (
        pd.DataFrame(
            rows,
            columns=[
                "series_id",
                "d_int",
                "quantile_level",
                "value"
            ]
        )
        .merge(dates, on="d_int", how="left")
    )

    out["model"] = "moirai_2"
    out = out[COLS]

    elapsed = time.perf_counter() - t0

    all_outputs.append(out)

    runtime_rows.append({
        "model": "moirai_2",
        "window": window,
        "seconds": elapsed
    })

    crossing_rows.append({
        "window": window,
        "forecast_points": n_points,
        "crossing_before": crossing_before,
        "crossing_before_pct": 100 * crossing_before / n_points,
        "crossing_after": crossing_after,
        "crossing_after_pct": 100 * crossing_after / n_points
    })

    print(
        f"{window}: {len(out):,} rows | "
        f"{elapsed:.1f} s | "
        f"crossing trước/sau: "
        f"{crossing_before:,}/{crossing_after:,}"
    )


# Ghép và kiểm tra kết quả
result = pd.concat(all_outputs, ignore_index=True)

expected_rows = len(windows) * len(ids) * H * len(Q)

assert len(result) == expected_rows
assert result["series_id"].nunique() == len(ids)
assert result["value"].notna().all()
assert np.isfinite(result["value"]).all()
assert (result["value"] >= 0).all()

wide = result.pivot(
    index=["series_id", "d_int"],
    columns="quantile_level",
    values="value"
)[sorted(Q)]

final_crossing = int(
    (np.diff(wide.to_numpy(), axis=1) < -1e-8)
    .any(axis=1)
    .sum()
)

assert final_crossing == 0


# Lưu output
output_dir = ROOT / "data/outputs/forecasts"
output_dir.mkdir(parents=True, exist_ok=True)

result.to_parquet(
    output_dir / "moirai_2.parquet",
    index=False
)

pd.DataFrame(runtime_rows).to_csv(
    output_dir / "moirai_2_runtime.csv",
    index=False
)


# Diagnostic crossing
crossing_df = pd.DataFrame(crossing_rows)

print("Moirai-2:", result.shape)

print("\nQuantile crossing trước/sau rearrangement:")
print(crossing_df.to_string(index=False))

total_points = crossing_df["forecast_points"].sum()
total_before = crossing_df["crossing_before"].sum()
total_after = crossing_df["crossing_after"].sum()

print(
    f"\nTổng crossing trước rearrangement: "
    f"{total_before:,}/{total_points:,} "
    f"({100 * total_before / total_points:.2f}%)"
)

print(
    f"Tổng crossing sau rearrangement  : "
    f"{total_after:,}/{total_points:,} "
    f"({100 * total_after / total_points:.2f}%)"
)
'''


result_moirai = subprocess.run(
    [str(MOIRAI_PYTHON), "-c", moirai_code],
    cwd=ROOT,
    capture_output=True,
    text=True
)

print(result_moirai.stdout)

if result_moirai.returncode != 0:
    print(result_moirai.stderr)

assert (
    result_moirai.returncode == 0
), "Moirai-2 full inference thất bại"


W1: 875,448 rows | 13.9 s | crossing trước/sau: 9,013/0
W2: 875,448 rows | 11.6 s | crossing trước/sau: 9,278/0
W3: 875,448 rows | 11.2 s | crossing trước/sau: 8,964/0
Moirai-2: (2626344, 6)

Quantile crossing trước/sau rearrangement:
window  forecast_points  crossing_before  crossing_before_pct  crossing_after  crossing_after_pct
    W1            97272             9013             9.265770               0                 0.0
    W2            97272             9278             9.538202               0                 0.0
    W3            97272             8964             9.215396               0                 0.0

Tổng crossing trước rearrangement: 27,255/291,816 (9.34%)
Tổng crossing sau rearrangement  : 0/291,816 (0.00%)



In [51]:
moirai = pd.read_parquet(PATH_OUTPUT / "moirai_2.parquet")
validate_forecast(moirai, "moirai_2")
check_quantile_crossing(moirai, "moirai_2")

wide = moirai.pivot(index=["series_id", "d_int"], columns="quantile_level", values="value")
print("q0.005 = q0.025:", f"{np.isclose(wide[0.005], wide[0.025]).mean():.2%}")
print("q0.975 = q0.995:", f"{np.isclose(wide[0.975], wide[0.995]).mean():.2%}")

moirai_2: 2,626,344 rows | 3,474 series | 9 quantiles
moirai_2: quantile crossing = 0
q0.005 = q0.025: 94.93%
q0.975 = q0.995: 98.90%


### Nhận xét kết quả Moirai-2

Moirai-2 tạo dự báo xác suất cho toàn bộ 3.474 chuỗi trong ba cửa sổ rolling-origin. Các phân vị native của checkpoint nằm trong khoảng 0.1–0.9; các mức phân vị yêu cầu nằm giữa các mức native được nội suy tuyến tính, trong khi các mức nằm ngoài phạm vi hỗ trợ được chặn tại hai biên.

Trước bước quantile rearrangement, một phần các điểm dự báo xuất hiện hiện tượng quantile crossing. Sau khi sắp xếp lại các phân vị tại từng điểm dự báo, không còn quantile crossing trong đầu ra cuối cùng. Vì vậy, tính đơn điệu của các phân vị cuối cùng một phần là kết quả của bước hậu xử lý, thay vì hoàn toàn là đặc tính của đầu ra native từ mô hình.

Ngoài ra, việc các phân vị cực biên bị chặn tại phạm vi hỗ trợ native làm hạn chế khả năng biểu diễn phần đuôi của phân phối. Đây là điểm cần lưu ý khi đánh giá chất lượng probabilistic forecast của Moirai-2.

## 07. Benchmark thời gian chạy

Thời gian chạy chỉ được so sánh khi cả ba mô hình được thực thi trong **cùng một phiên notebook, cùng phần cứng, cùng 3.474 chuỗi, cùng ba cửa sổ rolling-origin và cùng horizon dự báo**.

Runtime được đo cho quá trình tạo forecast trên từng cửa sổ; thời gian cài đặt dependency, khởi tạo môi trường và tải checkpoint không được tính vào benchmark.

Các mô hình có cơ chế tạo probabilistic forecast khác nhau, vì vậy runtime phản ánh chi phí của toàn bộ pipeline dự báo tương ứng trong thiết lập thực nghiệm này, không phải benchmark tốc độ tổng quát giữa các kiến trúc.

In [52]:
runtime = pd.concat([
    pd.read_csv(PATH_OUTPUT / "chronos_bolt_runtime.csv"),
    pd.read_csv(PATH_OUTPUT / "ttm_runtime.csv"),
    pd.read_csv(PATH_OUTPUT / "moirai_2_runtime.csv")
], ignore_index=True)

runtime_summary = runtime.groupby("model", as_index=False)["seconds"].agg(total_seconds="sum", avg_seconds_per_window="mean")
runtime_summary["total_minutes"] = runtime_summary["total_seconds"] / 60
runtime_summary = runtime_summary.sort_values("total_seconds").reset_index(drop=True)

display(runtime)
display(runtime_summary)


,model,window,seconds
0,chronos_bolt,W1,33.384728
1,chronos_bolt,W2,29.824345
2,chronos_bolt,W3,30.336468
3,ttm,W1,3.298635
4,ttm,W2,2.431146
5,ttm,W3,2.269791
6,moirai_2,W1,13.929780
7,moirai_2,W2,11.562453
8,moirai_2,W3,11.167386


,model,total_seconds,avg_seconds_per_window,total_minutes
0,ttm,7.999572,2.666524,0.133326
1,moirai_2,36.659619,12.219873,0.610994
2,chronos_bolt,93.545540,31.181847,1.559092


### Nhận xét benchmark

Bảng trên so sánh thời gian tạo forecast của ba Foundation Models trong cùng điều kiện thực nghiệm. Kết quả runtime cần được diễn giải cùng với sự khác biệt trong pipeline của từng mô hình: Chronos-Bolt tạo quantile forecast trực tiếp, TTM tạo point forecast rồi chuyển đổi sang phân vị bằng lớp bọc xác suất, trong khi Moirai-2 tạo probabilistic forecast và thực hiện thêm bước quantile mapping/rearrangement.

Do đó, thời gian chạy không được sử dụng độc lập để xác định mô hình tốt nhất. Việc lựa chọn mô hình cuối cùng cần kết hợp chi phí tính toán với các chỉ tiêu về độ chính xác và chất lượng dự báo xác suất ở bước đánh giá chung.

Các kết quả runtime chỉ đại diện cho môi trường phần cứng và cấu hình được sử dụng trong thí nghiệm này.

## 08. Rủi ro rò rỉ dữ liệu pre-training

Một vấn đề quan trọng khi đánh giá foundation model theo thiết lập zero-shot là khả năng dữ liệu đánh giá đã xuất hiện trong pre-training. Vì vậy, **zero-shot trong thí nghiệm này chỉ có nghĩa là không cập nhật trọng số bằng dữ liệu M5**, không đồng nghĩa tuyệt đối với *unseen during pre-training*.

### Chronos-Bolt

Model card Chronos-Bolt cho biết mô hình được huấn luyện trên gần 100 tỷ quan sát và các benchmark dùng trong đánh giá zero-shot của tác giả không được đưa vào training. Hệ sinh thái dữ liệu Chronos công khai có chứa M5, nhưng repository này bao gồm dữ liệu phục vụ cả training và evaluation. Vì vậy, sự hiện diện của M5 trong repository không đủ để kết luận checkpoint Chronos-Bolt đã được pre-train trên M5.

Nguồn: https://huggingface.co/amazon/chronos-bolt-base

### TinyTimeMixer

Model card của checkpoint TTM r2 công bố các nhóm dữ liệu pre-training và không liệt kê M5. Dựa trên thông tin công khai hiện có, không có bằng chứng cho thấy checkpoint TTM r2 sử dụng trực tiếp M5 trong pre-training. Tuy nhiên, đây không phải bằng chứng tuyệt đối rằng mọi dạng trùng lặp dữ liệu đều không tồn tại.

Nguồn: https://huggingface.co/ibm-granite/granite-timeseries-ttm-r2

### Moirai-2

Model card Moirai-2 mô tả pre-training bằng các phần không gây rò rỉ của GIFT-Eval, dữ liệu mixup từ các phần không gây rò rỉ của Chronos Dataset, dữ liệu tổng hợp và dữ liệu vận hành nội bộ của Salesforce. Tài liệu công khai vì vậy không cung cấp cơ sở để khẳng định Moirai-2 đã được pre-train trực tiếp trên phần M5 dùng trong nghiên cứu này.

Nguồn: https://huggingface.co/Salesforce/moirai-2.0-R-small

### Cách diễn giải

Hiệu năng tốt trên M5 không được xem là bằng chứng rằng mô hình đã ghi nhớ M5. Ngược lại, việc chạy zero-shot cũng không chứng minh tuyệt đối rằng không tồn tại contamination. Kết quả Foundation Models vì vậy cần được diễn giải cùng với giới hạn provenance của các corpus pre-training quy mô lớn.


## 09. Kiểm tra đầu ra cuối

Ba file raw forecast được xuất theo cùng schema 6 cột để chuyển cho bước hiệu chuẩn và đánh giá tiếp theo. Chronos giữ nguyên hạn chế tail clipping của checkpoint; TTM dùng NB/Poisson wrapper; Moirai-2 dùng quantile mapping và rearrangement để bảo đảm non-crossing.


In [53]:
forecast_files = {"chronos_bolt": PATH_OUTPUT / "chronos_bolt.parquet", "ttm": PATH_OUTPUT / "ttm.parquet", "moirai_2": PATH_OUTPUT / "moirai_2.parquet"}

for model_name, path in forecast_files.items():
    df = pd.read_parquet(path)
    validate_forecast(df, model_name)
    check_quantile_crossing(df, model_name)
    print(f"{model_name:<14} {path.name:<24} {path.stat().st_size / 1024**2:.1f} MB")

print("\n P4 hoàn tất: 3 Foundation Models đã có raw forecast hợp lệ.")


chronos_bolt: 2,626,344 rows | 3,474 series | 9 quantiles
chronos_bolt: quantile crossing = 0
chronos_bolt   chronos_bolt.parquet     10.1 MB
ttm: 2,626,344 rows | 3,474 series | 9 quantiles
ttm: quantile crossing = 0
ttm            ttm.parquet              1.3 MB
moirai_2: 2,626,344 rows | 3,474 series | 9 quantiles
moirai_2: quantile crossing = 0
moirai_2       moirai_2.parquet         13.3 MB

 P4 hoàn tất: 3 Foundation Models đã có raw forecast hợp lệ.


## Kết luận P4

Phần P4 đã triển khai ba Foundation Models gồm Chronos-Bolt, TinyTimeMixer và Moirai-2 theo cùng thiết lập rolling-origin trên mẫu dữ liệu M5. Đầu ra của cả ba mô hình được chuẩn hóa về cùng schema và 9 mức phân vị để phục vụ các bước hiệu chuẩn và đánh giá tiếp theo.

Ba mô hình đại diện cho các cơ chế dự báo xác suất khác nhau: Chronos-Bolt cung cấp quantile forecast trực tiếp, TTM sử dụng point forecast kết hợp lớp bọc Negative Binomial/Poisson, còn Moirai-2 sử dụng probabilistic forecast kết hợp quantile mapping và rearrangement. Những khác biệt này nên được xem xét và đánh giá khi so sánh kết quả giữa các mô hình.

Ở giai đoạn P4, việc kiểm tra tập trung vào tính hợp lệ của đầu ra, cơ chế tạo probabilistic forecast, chi phí tính toán và rủi ro pre-training leakage. Chưa thể đưa ra kết luận mô hình nào tốt nhất nếu chưa có đánh giá chung về độ chính xác và chất lượng dự báo xác suất.